In [1]:
import pandas as pd
import os
import networkx as nx
from tqdm import tqdm

In [8]:
ad_dir_path = '../../../data_preprocessed/grn_output/ad_nci_seperate_network/lioness/ad_preprocessed/'
nci_dir_path = '../../../data_preprocessed/grn_output/ad_nci_seperate_network/lioness/nci_preprocessed/'

In [7]:
networks_list_ad = []

for i in tqdm(os.listdir(ad_dir_path)):
    network = pd.read_csv(ad_dir_path+i)
    remove_genes = list(set(network['0']).intersection(network['1']))
    network = network[~network['1'].isin(remove_genes)]
    B = nx.Graph()
    B.add_nodes_from(network['0'].tolist(), bipartite = 'tf')
    B.add_nodes_from(network['1'].tolist(), bipartite = 'target_genes')
    for k,i in network.iterrows():
        #print()
        #print(i['1'])
        #print(i['2'])
        B.add_edge(i['0'],i['1'],weight = i['2'])

    networks_list_ad.append(B)
        


100%|███████████████████████████████████████████| 87/87 [11:43<00:00,  8.08s/it]


In [9]:
networks_list_nci = []

for i in tqdm(os.listdir(nci_dir_path)):
    network = pd.read_csv(nci_dir_path+i)
    remove_genes = list(set(network['0']).intersection(network['1']))
    network = network[~network['1'].isin(remove_genes)]
    B = nx.Graph()
    B.add_nodes_from(network['0'].tolist(), bipartite = 'tf')
    B.add_nodes_from(network['1'].tolist(), bipartite = 'target_genes')
    for k,i in network.iterrows():
        #print()
        #print(i['1'])
        #print(i['2'])
        B.add_edge(i['0'],i['1'],weight = i['2'])

    networks_list_nci.append(B)
        


100%|███████████████████████████████████████████| 67/67 [09:02<00:00,  8.10s/it]


In [11]:
import networkx as nx
import numpy as np
import glob

# Lists to store network metrics
ad_metrics = []
nci_metrics = []

def compute_network_metrics(G):
    """
    Compute various network properties for a given weighted bipartite gene regulatory network.
    """
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    
    # Calculate density
    regulators = {n for n, d in G.nodes(data=True) if d.get("bipartite", 0) == 'tf'}
    genes = set(G) - regulators
    max_possible_edges = len(regulators) * len(genes)
    density = num_edges / max_possible_edges if max_possible_edges > 0 else 0
    
    # Weighted Average degree
    avg_degree = (2 * sum(d for _, d in G.degree(weight='weight'))) / num_nodes if num_nodes > 0 else 0
    
    # Modularity (requires community detection, using greedy modularity method)
    from networkx.algorithms import community
    communities = list(community.greedy_modularity_communities(G))
    modularity = community.modularity(G, communities, weight='weight') if communities else 0
    
    # Weighted Clustering coefficient (bipartite version)
    clustering = nx.average_clustering(G, weight='weight')
    
    # Giant component size
    largest_cc = max(nx.connected_components(G), key=len, default=set())
    giant_component_size = len(largest_cc)
    
    # Weighted Nestedness (proxy: assortativity coefficient)
    nestedness = nx.degree_assortativity_coefficient(G, weight='weight') if num_nodes > 1 else 0
    
    return [density, avg_degree, modularity, clustering, giant_component_size, nestedness]

for network_ad in tqdm(networks_list_ad):
    ad_metrics.append(compute_network_metrics(network_ad))

# Load all Control networks and compute metrics


for network_nci in tqdm(networks_list_nci):
    nci_metrics.append(compute_network_metrics(network_nci))



 12%|████▍                                | 8/67 [4:28:28<32:59:56, 2013.51s/it]


KeyboardInterrupt: 

In [12]:
ad_metrics 

[[0.050483513814013024,
  749.6636927540494,
  0.19460735271476876,
  0.0,
  17181,
  -0.537007508568783],
 [0.04942237498879666,
  751.9397821765339,
  0.222183213927184,
  0.0,
  17178,
  -0.5175418626624856],
 [0.05101538393554434,
  747.9145103802688,
  0.17958843622612894,
  0.0,
  17173,
  -0.531943031079627],
 [0.04985934534509782,
  739.0807583500019,
  0.1995902140171478,
  0.0,
  17174,
  -0.5358630022944345],
 [0.05012474919527954,
  753.5202904595028,
  0.22559477610433173,
  0.0,
  17176,
  -0.5228558441675767],
 [0.04934398610744134,
  740.1790181917993,
  0.17329848079045473,
  0.0,
  17177,
  -0.5284782232961373],
 [0.05071930863450727,
  750.5141310466653,
  0.21191361800596045,
  0.0,
  17177,
  -0.5271388027045799],
 [0.05038364707006623,
  830.1465786735979,
  0.2826327945857228,
  0.0,
  17253,
  -0.41332650914805874],
 [0.048205547185386874,
  761.5600249960735,
  0.19710626248682916,
  0.0,
  17183,
  -0.48685175838653677],
 [0.04981100784769841,
  744.8704976489

In [ ]:
# Convert results to NumPy arrays for further analysis
ad_metrics = np.array(ad_metrics)

# Print sample output
print("AD Metrics (first 5 networks):\n", ad_metrics[:5])
print("Control Metrics (first 5 networks):\n", nci_metrics[:5])